In [ ]:
#| default_exp slack_bot

# Step 11: Submit a bird from Slack

**Goal:** anyone in your Slack can post a bird photo in `#app-bird-watcher`, the bot identifies it with the lesson 4/5 pipeline, saves it under *their name*, and replies with a link to *their* personal gallery.

Until now our Slack app could only **talk** (the webhook from lesson 7). A bot can also **listen** — and listening changes the plumbing. A webhook is a URL Slack lets *us* post to; to hear messages, Slack needs to reach *our* code. Instead of opening our home network to the internet, we use **Socket Mode**: the bot dials *out* to Slack and holds a websocket open, and events flow back down that pipe. Like a phone call you place — nobody needs your number.

One new dependency (`slack-bolt`, Slack's official bot framework), one new process (`bird-watcher-bot`), zero new ideas about storage or identification — those come from lessons 4, 5, and 6 unchanged.

## Step 11.0 — Upgrade the Slack app

The manifest at [`setup/slack-manifest.yml`](../setup/slack-manifest.yml) now describes a real bot. On [api.slack.com/apps](https://api.slack.com/apps), open your app → **App Manifest** → paste the new version → save, then:

1. **Reinstall** the app to your workspace (Slack asks again because the permissions grew).
2. **OAuth & Permissions** → copy the *Bot User OAuth Token* (`xoxb-...`) into `.env` as `SLACK_BOT_TOKEN`.
3. **Basic Information → App-Level Tokens** → *Generate Token* with the `connections:write` scope → copy it (`xapp-...`) into `.env` as `SLACK_APP_TOKEN`.
4. In `#app-bird-watcher`, type `/invite @Bird Watcher` — bots only hear channels they've been invited to.

Three secrets now, one job each: the *webhook* posts camera alerts to one channel, the *bot token* lets the bot read files and reply, the *app token* opens the websocket.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os

assert load_dotenv(), "Failed to load .env file"

SLACK_BOT_TOKEN = os.environ.get("SLACK_BOT_TOKEN", "")
SLACK_APP_TOKEN = os.environ.get("SLACK_APP_TOKEN", "")
GALLERY_URL = os.environ.get("GALLERY_URL", "http://localhost:8123")

DATA_FOLDER = Path("../data")
DB_PATH = DATA_FOLDER / "birds.db"

print(f"bot token set: {bool(SLACK_BOT_TOKEN)} | app token set: {bool(SLACK_APP_TOKEN)}")

## Step 11.1 — What already changed under our feet

Two earlier lessons grew to support this one — go look, both changes are small:

- **Lesson 6:** `Sighting` gained `submitted_by` (a Slack user id; `None` still means the backyard camera), the table gained a matching column — including a tiny **migration** so old databases upgrade themselves — and `get_sightings` filters by it, using the same `WHERE`-list pattern as `species`.
- **Lesson 9:** `/gallery` takes `?submitted_by=`, and `gallery.html` reads `?user=` from its own URL and passes it along. So `/gallery.html?user=U0123` is someone's personal gallery.

One honest caveat, worth saying out loud: that link is **identification, not authentication**. Anyone who has it can see that view — fine for a family bird page, not how a bank works. Real login is a whole other lesson.

## Step 11.2 — `identify_and_record`: the whole pipeline, one function

Photo bytes go in, a Slack-ready reply comes out. In between: the YOLO detector from lesson 4, the species classifier from lesson 5, and the storage contract from lesson 6.

Two decisions to notice:

- **Submissions skip the cooldown** and call `post_sighting` directly. The cooldown exists so the *camera* doesn't count one loitering cardinal ten times; a human pressing send is always intentional.
- **The pipeline speaks file paths** (`AnnotatedImage.from_path`), so we park the bytes in a temporary file. `tempfile.TemporaryDirectory()` cleans up after us — same `with` habit as lesson 6's database connections.

In [ ]:
#| export
import tempfile
from pathlib import Path

from ultralytics import YOLO

from bird_watcher.identify import ImageClassifierEnum  # noqa: F401 — importing wires classifiers onto detections (lesson 5)
from bird_watcher.detect import AnnotatedImage
from bird_watcher.persistence import Sighting, SightingStorage


def identify_and_record(
    image_bytes: bytes,
    submitted_by: str,
    storage: SightingStorage,
    model: YOLO,
    confidence: float = 0.5,
    gallery_url: str = "http://localhost:8123",
) -> str:
    """Identify the bird(s) in a submitted photo, save them, and write the reply.

    Args:
        image_bytes: the photo, as downloaded from Slack.
        submitted_by: the Slack user id of whoever sent it.
        storage: any SightingStorage.
        model: the YOLO detector from lesson 4.
        confidence: minimum detection confidence.
        gallery_url: base URL for the personal-gallery link in the reply.

    Returns:
        The message to send back to the submitter.
    """
    with tempfile.TemporaryDirectory() as folder:
        img_path = Path(folder) / "submission.jpg"
        img_path.write_bytes(image_bytes)

        annotated = AnnotatedImage.from_path(img_path=str(img_path), model=model, confidence=confidence)
        identified = [d for d in annotated.detections if d.Classifier is not None]

        if not identified:
            return "I looked, but couldn't find a bird in that photo. Try a closer shot?"

        lines = []
        for detection in identified:
            sighting = Sighting.from_detection(detection)
            sighting.submitted_by = submitted_by

            storage.post_sighting(sighting)  # a human pressed send — no cooldown

            lines.append(f"\u2022 *{sighting.species}* ({sighting.confidence:.0%} sure)")

    link = f"{gallery_url}/gallery.html?user={submitted_by}"
    return "I spotted:\n" + "\n".join(lines) + f"\n<{link}|See your gallery>"

### Try it — no Slack required

Same trick as always: a fake storage, a sample photo from lesson 4, and the real detector. If this cell passes, the *entire* submission pipeline works — Slack is just delivery.

In [ ]:
from dataclasses import dataclass, field

from ultralytics import YOLO

from bird_watcher.detect import YoloModelEnum
from bird_watcher.persistence import Sighting, SightingStorage
from bird_watcher.slack_bot import identify_and_record


@dataclass
class FakeSightingStorage(SightingStorage):
    sightings: list[Sighting] = field(default_factory=list)

    def post_sighting(self, sighting, debug_prn=False):
        sighting.id = len(self.sightings) + 1
        self.sightings.insert(0, sighting)

    def get_sightings(self, species=None, limit=100, offset=0, submitted_by=None):
        rows = [
            s for s in self.sightings
            if (species is None or s.species == species)
            and (submitted_by is None or s.submitted_by == submitted_by)
        ]
        return rows[offset:offset + limit]

    def get_sighting(self, sighting_id):
        return next((s for s in self.sightings if s.id == sighting_id), None)


model = YOLO(YoloModelEnum.QUALITY.value)
fake = FakeSightingStorage(cooldown_minutes=0)

image_bytes = (DATA_FOLDER / "samples/dog_bird.jpg").read_bytes()
reply = identify_and_record(image_bytes, submitted_by="U_TEST_NEPHEW", storage=fake, model=model, confidence=0.7)

print(reply)
assert fake.sightings, "should have recorded at least one sighting"
assert all(s.submitted_by == "U_TEST_NEPHEW" for s in fake.sightings)
assert "gallery.html?user=U_TEST_NEPHEW" in reply
print(f"\n✅ pipeline recorded {len(fake.sightings)} sighting(s) for U_TEST_NEPHEW")

## Step 11.3 — The bot: a factory, again

`create_bot` mirrors lesson 9's `create_app` on purpose — a factory that takes the storage contract, so the bot doesn't know which database sits behind it either.

The handler does three things and refuses to be clever:

1. **Ignore anything from a bot** (`bot_id` check) — including ourselves. This is infinite-loop insurance: our reply is a message too.
2. **Ignore messages without images** — the channel stays a normal channel.
3. For each photo: download it (with the bot token — Slack files are private), run `identify_and_record`, and reply **in a thread** under the photo, so identifications don't bury the conversation.

In [ ]:
#| export
import requests
from slack_bolt import App


def create_bot(
    storage: SightingStorage,
    model: YOLO,
    bot_token: str,
    gallery_url: str = "http://localhost:8123",
) -> App:
    """Build the Slack bot: photo in, species out.

    Args:
        storage: any SightingStorage — same contract as the web app.
        model: the YOLO detector.
        bot_token: xoxb-... token; used to download files and send replies.
        gallery_url: base URL for the personal-gallery links.

    Returns:
        A configured slack_bolt App, ready for a SocketModeHandler.
    """
    bot = App(token=bot_token)

    @bot.event("message")
    def handle_message(event, say):
        if event.get("bot_id"):
            return  # never answer a bot — especially not ourselves

        images = [f for f in event.get("files", []) if f.get("mimetype", "").startswith("image/")]
        if not images:
            return  # plain chatter — stay quiet

        for file in images:
            response = requests.get(
                file["url_private_download"],
                headers={"Authorization": f"Bearer {bot_token}"},  # Slack files are private
                timeout=30,
            )
            response.raise_for_status()

            reply = identify_and_record(
                image_bytes=response.content,
                submitted_by=event["user"],
                storage=storage,
                model=model,
                gallery_url=gallery_url,
            )
            say(text=reply, thread_ts=event["ts"])  # reply in a thread, under the photo

    return bot

## Step 11.4 — Run it: the third entrypoint

Your app is now three one-command processes, all reading the same database and the same environment variables:

| command | job |
|---|---|
| `uv run bird-watcher-web` | serve the gallery |
| `uv run bird-watcher-digest` | one Slack digest per run (cron calls it daily) |
| `uv run bird-watcher-bot` | listen for photo submissions |

Run the bot from a terminal (not a notebook cell — it blocks forever, like the web server would):

```bash
uv run bird-watcher-bot
```

Then post a bird photo in `#app-bird-watcher` and watch it answer in a thread.

In [ ]:
#| export
import os


def main():
    """Start the Bird Watcher Slack bot — listens until you stop it.

    Socket Mode: the bot dials OUT to Slack, so it runs from home WiFi with
    no public URL. Settings come from environment variables:

        SLACK_BOT_TOKEN  — xoxb-... (OAuth & Permissions page)
        SLACK_APP_TOKEN  — xapp-... (App-Level Token with connections:write)
        BIRDS_DB         — path to the SQLite file   (default: data/birds.db)
        GALLERY_URL      — base URL for reply links  (default: http://localhost:8123)
    """
    from slack_bolt.adapter.socket_mode import SocketModeHandler

    from bird_watcher.detect import YoloModelEnum
    from bird_watcher.persistence import SqliteSightingStorage

    bot_token = os.environ["SLACK_BOT_TOKEN"]  # crash loudly if missing — the bot is useless without them
    app_token = os.environ["SLACK_APP_TOKEN"]

    storage = SqliteSightingStorage(
        db_path=os.environ.get("BIRDS_DB", "data/birds.db"),
        cooldown_minutes=0,  # submissions skip the cooldown anyway
    )
    model = YOLO(YoloModelEnum.QUALITY.value)

    bot = create_bot(
        storage,
        model,
        bot_token=bot_token,
        gallery_url=os.environ.get("GALLERY_URL", "http://localhost:8123"),
    )

    print("Bird Watcher bot is listening — post a bird photo in your channel.")
    SocketModeHandler(bot, app_token).start()

## Acceptance criterion

- The pipeline test above records sightings tagged `U_TEST_NEPHEW` and the reply contains their personal gallery link.
- The gallery API honors the filter: the cell below proves each user sees only their own birds.
- With tokens in `.env`: `uv run bird-watcher-bot`, post a photo in `#app-bird-watcher`, get a threaded identification back, and the link it sends shows just your birds.

In [ ]:
from fastapi.testclient import TestClient
from bird_watcher.web_app import create_app

client = TestClient(create_app(fake))

mine = client.get("/gallery?submitted_by=U_TEST_NEPHEW").json()
everyone = client.get("/gallery").json()
nobody = client.get("/gallery?submitted_by=U_SOMEONE_ELSE").json()

assert mine["count"] == len(fake.sightings)
assert nobody["count"] == 0
assert everyone["count"] >= mine["count"]
assert all(s["submitted_by"] == "U_TEST_NEPHEW" for s in mine["sightings"])

print(f"✅ per-user gallery works. Personal link: {GALLERY_URL}/gallery.html?user=U_TEST_NEPHEW")

## 🎉 The bird watcher is a team sport now

The camera watches the backyard; your friends watch the world. Everything lands in one database, everyone gets their own gallery, and the pieces still don't know about each other: storage speaks the contract, the web app speaks the contract, the bot speaks the contract.

**Ideas to steal next:** a `/birds today` slash command (the digest, on demand), reacting with 🐦 instead of a thread reply, or a leaderboard endpoint — who's submitted the most species?